In [ ]:
# ============================================================
# Cell 1: Imports
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print("Imports successful")

In [ ]:
# ============================================================
# Cell 2: Dataset Configuration
# ============================================================

N_SAMPLES = 10_000

# Percentage of observations that are censored
CENSORING_RATE = 0.0

# Small random noise added to the true survival time
TIME_NOISE = 0.01

print(f"Samples: {N_SAMPLES:,}")
print(f"Censoring rate: {CENSORING_RATE:.0%}")
print(f"Time noise: {TIME_NOISE}")

In [ ]:
# ============================================================
# Cell 3: Generate Features
# ============================================================

rng = np.random.default_rng(RANDOM_STATE)

df = pd.DataFrame({
    "age": rng.uniform(18, 80, N_SAMPLES),
    "mileage": rng.uniform(5_000, 300_000, N_SAMPLES),
    "missed_services": rng.integers(0, 10, N_SAMPLES),
    "ambient_temp": rng.uniform(15, 50, N_SAMPLES),
    "engine_cc": rng.choice(
        [1200, 1500, 1800, 2000, 2500, 3000, 3500],
        N_SAMPLES
    ),
    "weight_kg": rng.uniform(900, 2500, N_SAMPLES),
    "driving_stress": rng.uniform(0, 1, N_SAMPLES),
})

df.head()

In [ ]:
# ============================================================
# Cell 4: Create Ground-Truth Risk Function
# ============================================================

age = df["age"].values
mileage = df["mileage"].values
missed_services = df["missed_services"].values
temperature = df["ambient_temp"].values
engine_cc = df["engine_cc"].values
weight = df["weight_kg"].values
driving_stress = df["driving_stress"].values


# ------------------------------------------------------------
# Normalize continuous variables
# ------------------------------------------------------------

age_n = (age - 18) / (80 - 18)

mileage_n = (mileage - 5_000) / (300_000 - 5_000)

temp_n = (temperature - 15) / (50 - 15)

weight_n = (weight - 900) / (2500 - 900)

engine_n = (engine_cc - 1200) / (3500 - 1200)


# ------------------------------------------------------------
# Base risk
# ------------------------------------------------------------

risk = (
    1.0 * age_n
    + 2.0 * mileage_n
    + 1.5 * (missed_services / 10)
    + 0.8 * temp_n
    + 0.8 * driving_stress
)


# ------------------------------------------------------------
# Nonlinear effects
# ------------------------------------------------------------

risk += 2.0 * age_n ** 2

risk += 1.5 * mileage_n ** 2

risk += 2.5 * np.maximum(mileage_n - 0.65, 0)

risk += 1.5 * np.maximum(age_n - 0.70, 0)


# ------------------------------------------------------------
# Interactions
# ------------------------------------------------------------

risk += 2.0 * mileage_n * age_n

risk += 1.5 * missed_services / 10 * mileage_n

risk += 1.0 * driving_stress * age_n

risk += 1.0 * temp_n * engine_n


# ------------------------------------------------------------
# Vehicle weight / engine effects
# ------------------------------------------------------------

risk += 0.5 * weight_n

risk += 0.4 * engine_n


# ------------------------------------------------------------
# Ensure risk is positive
# ------------------------------------------------------------

risk = risk - risk.min() + 0.1

df["true_risk"] = risk

df["true_risk"].describe()

In [ ]:
# ============================================================
# Cell 5: Generate Survival Time
# ============================================================

# Higher risk -> shorter survival time
#
# The exponential transformation makes the relationship
# strongly nonlinear.

base_time = 5000 * np.exp(-1.5 * df["true_risk"].values)


# Small amount of random noise
noise = rng.normal(
    loc=0,
    scale=TIME_NOISE,
    size=N_SAMPLES
)

duration = base_time * (1 + noise)


# Ensure all times are positive
duration = np.maximum(duration, 1.0)

df["duration_days"] = duration

df["duration_days"].describe()

In [ ]:
# ============================================================
# Cell 6: Generate Event Indicator
# ============================================================

df["event"] = 1

print("Events:")
print(df["event"].value_counts())

print("\nEvent rate:")
print(df["event"].mean())

In [ ]:
# ============================================================
# Cell 7: Final Dataset
# ============================================================

feature_columns = [
    "age",
    "mileage",
    "missed_services",
    "ambient_temp",
    "engine_cc",
    "weight_kg",
    "driving_stress"
]

target_columns = [
    "duration_days",
    "event"
]

final_columns = feature_columns + target_columns

dataset = df[final_columns].copy()

dataset.head()

In [ ]:
# ============================================================
# Cell 8: Dataset Inspection
# ============================================================

print("Shape:")
print(dataset.shape)

print("\nMissing values:")
print(dataset.isnull().sum())

print("\nData types:")
print(dataset.dtypes)

print("\nSummary:")
display(dataset.describe())

In [ ]:
# ============================================================
# Cell 9: Visualize Ground Truth
# ============================================================

plt.figure(figsize=(10, 6))

plt.scatter(
    dataset["mileage"],
    dataset["duration_days"],
    alpha=0.25,
    s=10
)

plt.xlabel("Mileage")
plt.ylabel("Survival Time (days)")
plt.title("Mileage vs Survival Time")

plt.show()

In [ ]:
# ============================================================
# Cell 10: Correlation with Survival Time
# ============================================================

correlations = (
    dataset
    .corr(numeric_only=True)["duration_days"]
    .sort_values()
)

print(correlations)

In [ ]:
dataset.loc[0:1,"event"] = 0

In [ ]:
dataset

In [ ]:
# ============================================================
# Cell 11: Train/Test Split
# ============================================================

X = dataset[feature_columns]

y = Surv.from_dataframe(
    event="event",
    time="duration_days",
    data=dataset
)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# ============================================================
# Cell 12: Random Survival Forest
# ============================================================

rsf = RandomSurvivalForest(
    n_estimators=200,
    min_samples_split=6,
    min_samples_leaf=3,
    max_features="sqrt",
    max_depth=20,
    n_jobs=1,
    random_state=RANDOM_STATE
)

print(rsf)

In [ ]:
# ============================================================
# Cell 13: Train RSF
# ============================================================

print("Training RSF...")

rsf.fit(
    X_train,
    y_train
)

print("Training complete.")

In [ ]:
# ============================================================
# Cell 14: Generate Risk Predictions
# ============================================================

train_risk = rsf.predict(X_train)
test_risk = rsf.predict(X_test)

print("Train predictions:", train_risk.shape)
print("Test predictions:", test_risk.shape)

print("\nExample predictions:")
print(test_risk[:10])

In [ ]:
# ============================================================
# Cell 15: Evaluate C-index
# ============================================================

train_cindex = concordance_index_censored(
    y_train["event"],
    y_train["duration_days"],
    train_risk
)[0]

test_cindex = concordance_index_censored(
    y_test["event"],
    y_test["duration_days"],
    test_risk
)[0]


print(f"Training C-index: {train_cindex:.4f}")
print(f"Testing C-index:  {test_cindex:.4f}")

In [ ]:
# ============================================================
# Cell 16: Survival Function Predictions
# ============================================================

survival_functions = rsf.predict_survival_function(
    X_test.iloc[:5],
    return_array=True
)

print("Shape:")
print(survival_functions.shape)

In [ ]:
# ============================================================
# Cell 17: Predicted Failure Time
# ============================================================

time_points = rsf.unique_times_

predicted_failure_times = []

for survival_curve in survival_functions:

    below_50 = np.where(survival_curve <= 0.5)[0]

    if len(below_50) == 0:
        predicted_failure_times.append(np.inf)
    else:
        predicted_failure_times.append(
            time_points[below_50[0]]
        )

predicted_failure_times = np.array(
    predicted_failure_times
)

results = X_test.iloc[:5].copy()

results["predicted_failure_days"] = predicted_failure_times
results["actual_failure_days"] = y_test["duration_days"][:5]

results

In [ ]:
# ============================================================
# Cell 18: Predicted vs Actual Survival Time
# ============================================================

all_survival_functions = rsf.predict_survival_function(
    X_test,
    return_array=True
)

predicted_failure_times = []

for survival_curve in all_survival_functions:

    below_50 = np.where(survival_curve <= 0.5)[0]

    if len(below_50) == 0:
        predicted_failure_times.append(np.inf)
    else:
        predicted_failure_times.append(
            time_points[below_50[0]]
        )

predicted_failure_times = np.array(
    predicted_failure_times
)

actual_failure_times = y_test["duration_days"]


finite_mask = np.isfinite(predicted_failure_times)

plt.figure(figsize=(8, 8))

plt.scatter(
    actual_failure_times[finite_mask],
    predicted_failure_times[finite_mask],
    alpha=0.25,
    s=10
)

min_value = min(
    actual_failure_times[finite_mask].min(),
    predicted_failure_times[finite_mask].min()
)

max_value = max(
    actual_failure_times[finite_mask].max(),
    predicted_failure_times[finite_mask].max()
)

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Failure Time")
plt.ylabel("Predicted Failure Time")
plt.title("RSF: Predicted vs Actual Failure Time")

plt.show()

In [ ]:
# ============================================================
# Cell 19: Feature Importance
# ============================================================

importance = pd.Series(
    rsf.feature_importances_,
    index=feature_columns
).sort_values(ascending=False)

print(importance)

In [ ]:
# ============================================================
# Cell 20: Feature Importance Plot
# ============================================================

plt.figure(figsize=(10, 6))

importance.sort_values().plot(
    kind="barh"
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Survival Forest Feature Importance")

plt.show()

In [ ]:
# ============================================================
# Cell 21: Ground Truth Benchmark
# ============================================================

true_risk_test = df.loc[
    X_test.index,
    "true_risk"
].values

actual_time_test = y_test["duration_days"]

ground_truth_cindex = concordance_index_censored(
    y_test["event"],
    y_test["duration_days"],
    true_risk_test
)[0]

print(
    f"Ground-truth C-index: {ground_truth_cindex:.4f}"
)